In [ ]:
DATAFRAME df

# **ARIMA**

In [ ]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

model_arima = ARIMA(df['Price'], order=(1, 1, 1))
results_arima = model_arima.fit()

# Forecast 10 steps ahead
forecast = results_arima.get_forecast(steps=10)
forecast_mean = forecast.predicted_mean
forecast_ci   = forecast.conf_int(alpha=0.05)   # 95% confidence interval

print(forecast_mean)
print(forecast_ci)

NameError: name 'df' is not defined

# **GARCH**

In [ ]:
from arch import arch_model
import numpy as np

df['Returns'] = 100 * df['Price'].pct_change()

model_garch = arch_model(df['Returns'].dropna(), vol='Garch', p=1, q=1)
results_garch = model_garch.fit(disp='off')

# Forecast variance 10 steps ahead
forecast_garch = results_garch.forecast(horizon=10, reindex=False)

# Conditional variance forecast (σ²)
variance_forecast = forecast_garch.variance.iloc[-1]

# Convert to annualised volatility (assuming daily returns)
vol_forecast = np.sqrt(variance_forecast) * np.sqrt(252)

print("Variance forecast:\n", variance_forecast)
print("\nAnnualized volatility forecast:\n", vol_forecast)

# **ARX**

In [ ]:
# We need future DXY values to forecast — we'll use actuals
future_dxy = df['DXY'].iloc[-10:].values.reshape(-1, 1)   # placeholder: last 10 known values

model_arx = ARIMA(df['Price'], exog=df['DXY'], order=(1, 0, 0))
results_arx = model_arx.fit()

forecast_arx = results_arx.get_forecast(steps=10, exog=future_dxy)
print(forecast_arx.predicted_mean)
print(forecast_arx.conf_int())

# **VAR**
# Key assumptions and limitations:

**Stationarity:** VAR requires all variables to be stationary. You typically difference prices into returns first. If variables are non-stationary but cointegrated, you should use a VECM (Vector Error Correction Model) instead, which adds a long-run equilibrium term.


**Lag selection:** too few lags and you miss dynamics; too many and you overfit. Use AIC, BIC, or HQIC to select. BIC tends to select more parsimonious models.


**Curse of dimensionality:** a VAR with k variables and p lags estimates k^2*p coefficients plus constants. With 5 variables and 4 lags that's already 100 parameters. Keep the system small or use regularised variants (LASSO-VAR, Bayesian VAR).

**Identification:** the reduced-form VAR doesn't tell you about structural shocks. If you want to make causal claims (e.g. "a Fed rate shock causes this equity response"), you need a Structural VAR (SVAR) with additional identifying restrictions.

In [ ]:
from statsmodels.tsa.api import VAR
import pandas as pd

# Both series need to be stationary — difference if needed
data = df[['Returns', 'DXY_Returns']].dropna()

model_var = VAR(data)

# Select lag order by information criterion
lag_order = model_var.select_order(maxlags=10)
print(lag_order.summary())

# Fit with chosen lag (e.g. AIC-selected)
p = lag_order.aic
results_var = model_var.fit(p)
print(results_var.summary())

# Forecast 10 steps ahead — no need to supply future X, it's endogenous
forecast = results_var.forecast(data.values[-p:], steps=10)
forecast_df = pd.DataFrame(forecast, columns=['Returns_forecast', 'DXY_forecast'])
print(forecast_df)

In [ ]:
#Granger Causality
results_var.test_causality('Returns', 'DXY_Returns', kind='f').summary()

In [ ]:
#Impulse Response Functions (IRF)
irf = results_var.irf(periods=20)
irf.plot(orth=False)

In [ ]:
#Forecast Error Variance Decomposition (FEVD)
fevd = results_var.fevd(periods=20)
fevd.plot()